# MammosighTR — YOLO11m-cls ile BI-RADS Sınıflandırması

Bu çalışma, MammosighTR veri seti üzerinde dikkat mekanizmalı **YOLO11m-cls** (C3k2 ve C2PSA blokları) derin öğrenme modelinin iki aşamalı progresif çözünürlük (FixRes) eğitimi, çok açılı test zamanı artırımı (TTA) ve hasta düzeyinde çoklu projeksiyon bağlamsal füzyon değerlendirmesini içermektedir.

### Metodoloji ve Mimari Detaylar:
1. **İki Aşamalı Progresif Eğitim (FixRes):** 512x512 çözünürlükte ana eğitim ve 576x576 çözünürlükte yüksek çözünürlük ince ayarı (fine-tuning).
2. **Çok Açılı Test Zamanı Artırımı (6-View TTA):** Yatay çevirme ve çoklu ölçek çıkarımlarının ağırlıklı birleşimi.
3. **Hasta Düzeyinde Bağlamsal Füzyon:** Radyolojik değerlendirme pratiğine uygun olarak tekil projeksiyonların bilateral ve kraniokaudal/mediolateral oblik (RCC, LCC, RMLO, LMLO) bağlam entegrasyonu.
4. **Pareto-Optimal Karar Kalibrasyonu:** Yüksek riskli BI-RADS 4-5 sınıfı için duyarlılık ve makro F1 skorunu optimize eden klinik eşik kalibrasyonu.


In [ ]:
# ==============================================================================
# Bölüm 1: Ortam Konfigürasyonu, Bağımlılıklar ve Başlatma
# ==============================================================================
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'scipy', 'opencv-python-headless'], check=False)

import os, time, json, shutil, zipfile, glob, re
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

from ultralytics import YOLO
from scipy.optimize import minimize
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, roc_auc_score, f1_score, accuracy_score
)
from sklearn.preprocessing import label_binarize
import warnings
warnings.filterwarnings('ignore')

SESSION_START_TIME = time.time()
MAX_SESSION_HOURS  = 9.8   # Kaggle 12 saat sınırından önce güvenli kayıt ve çıkış süresi

print(f'Kurulum tamamlandı. PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()} (GPU: {torch.cuda.device_count()})')


In [ ]:
# ==============================================================================
# Bölüm 2: Veri Seti Arşivinin Doğrulanması ve Kontrol Noktalarının Tespiti
# ==============================================================================
possible_paths = [
    '/kaggle/input/mammosight-dataset-prep/yolo_dataset.zip',
    '/kaggle/working/yolo_dataset.zip',
    './yolo_dataset.zip'
]
found = glob.glob('/kaggle/input/**/yolo_dataset.zip', recursive=True)
if found:
    possible_paths = found + possible_paths

ZIP_PATH = next((p for p in possible_paths if os.path.exists(p)), None)
if ZIP_PATH is None:
    raise FileNotFoundError(
        "yolo_dataset.zip bulunamadı! Sağ taraftaki '+ Add Input' butonuyla "
        "'mammosight_dataset_prep' notebook çıktısını ekleyin."
    )

print(f'Veri seti bulundu: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1024/1024:.1f} MB)')

DATASET_DIR = '/kaggle/working/dataset' if os.path.exists('/kaggle') else './dataset'
if not os.path.exists(os.path.join(DATASET_DIR, 'test')):
    os.makedirs(DATASET_DIR, exist_ok=True)
    print('Zip çıkarılıyor...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(DATASET_DIR)

CLASS_NAMES   = ['BIRADS0', 'BIRADS1_2', 'BIRADS4_5']
DISPLAY_NAMES = ['BI-RADS 0', 'BI-RADS 1-2', 'BI-RADS 4-5']

for split in ['train', 'val', 'test']:
    counts = {cls: len(glob.glob(os.path.join(DATASET_DIR, split, cls, '*.jpg'))) for cls in CLASS_NAMES}
    print(f'{split.upper():5s}: {counts} (Toplam: {sum(counts.values())})')


In [ ]:
# ==============================================================================
# Bölüm 3: İki Aşamalı Progresif Çözünürlük Eğitimi (FixRes) ve Kontrol Noktası Yönetimi
# ==============================================================================
MODEL_NAME     = 'yolo11m-cls'
EPOCHS_STAGE1  = 50
EPOCHS_STAGE2  = 12
IMG_SIZE       = 512
IMG_SIZE_HIRES = 576
BATCH          = 16
WORKERS        = 4
PROJECT        = '/kaggle/working/runs' if os.path.exists('/kaggle') else './runs'
RUN_STAGE1     = 'mammosight_yolo11_stage1'
RUN_STAGE2     = 'mammosight_yolo11_stage2_576'
CACHE_NPZ      = os.path.join(PROJECT, 'yolo11_tta_cache.npz')
DEVICE         = [0, 1] if torch.cuda.device_count() > 1 else (0 if torch.cuda.is_available() else 'cpu')
os.makedirs(PROJECT, exist_ok=True)

# Çalışma dizini kontrolü: Önceki eğitim aşamaları ve kontrol noktalarının yüklenmesi
if os.path.exists('/kaggle/input'):
    for run_folder in [RUN_STAGE1, RUN_STAGE2]:
        target_dir = os.path.join(PROJECT, run_folder)
        if not os.path.exists(os.path.join(target_dir, 'weights', 'last.pt')):
            matches = glob.glob(f'/kaggle/input/**/{run_folder}/weights/*.pt', recursive=True)
            if matches:
                src_run_dir = os.path.dirname(os.path.dirname(matches[0]))
                print(f'Önceki oturumdan {run_folder} bulundu, kopyalanıyor: {src_run_dir}')
                shutil.copytree(src_run_dir, target_dir, dirs_exist_ok=True)
    if not os.path.exists(CACHE_NPZ):
        found_npz = glob.glob(f'/kaggle/input/**/yolo11_tta_cache.npz', recursive=True)
        if found_npz:
            shutil.copy2(found_npz[0], CACHE_NPZ)
            print(f'Önceki oturumdan TTA önbelleği bulundu: {CACHE_NPZ}')

def is_stage_completed(run_dir, target_epochs):
    state_file = os.path.join(run_dir, 'stage_state.json')
    best_pt    = os.path.join(run_dir, 'weights', 'best.pt')
    res_csv    = os.path.join(run_dir, 'results.csv')
    if os.path.exists(state_file) and os.path.exists(best_pt):
        try:
            st = json.load(open(state_file))
            if st.get('completed', False):
                return True
        except Exception:
            pass
    if os.path.exists(res_csv) and os.path.exists(best_pt):
        try:
            df_r = pd.read_csv(res_csv)
            if len(df_r) >= target_epochs:
                return True
        except Exception:
            pass
    return False

def save_stage_state(run_dir, epoch, completed):
    if os.environ.get('RANK', '-1') not in ('-1', '0') or os.environ.get('LOCAL_RANK', '-1') not in ('-1', '0'):
        return
    try:
        os.makedirs(run_dir, exist_ok=True)
        state_file = os.path.join(run_dir, 'stage_state.json')
        tmp_file = f'{state_file}.{os.getpid()}.tmp'
        with open(tmp_file, 'w') as f:
            json.dump({'epoch': int(epoch), 'completed': bool(completed), 'timestamp': time.time()}, f, indent=2)
        os.replace(tmp_file, state_file)
    except Exception:
        pass

session_timed_out = False

def make_epoch_callback(run_dir, target_epochs):
    def _cb(trainer):
        global session_timed_out
        ep = int(trainer.epoch) + 1
        done = (ep >= target_epochs)
        save_stage_state(run_dir, ep, done)
        elapsed_h = (time.time() - SESSION_START_TIME) / 3600.0
        if elapsed_h >= MAX_SESSION_HOURS:
            print(f'\n Oturum süresi sınırı ({MAX_SESSION_HOURS} saat) doldu! Epoch {ep} kaydedildi, eğitim güvenle durduruluyor...')
            session_timed_out = True
            trainer.stop = True
    return _cb

# Aşama 1: 512x512 Temel Temsil Eğitimi
s1_dir  = os.path.join(PROJECT, RUN_STAGE1)
s1_last = os.path.join(s1_dir, 'weights', 'last.pt')
s1_best = os.path.join(s1_dir, 'weights', 'best.pt')

if is_stage_completed(s1_dir, EPOCHS_STAGE1):
    print(f'AŞAMA 1 önceki oturumda tamamlanmış! Eğitim atlanıyor -> {s1_best}')
else:
    resuming_s1 = os.path.exists(s1_last)
    init_weights_s1 = s1_last if resuming_s1 else f'{MODEL_NAME}.pt'
    print(f'Aşama 1 başlatılıyor (Resume={resuming_s1}): {init_weights_s1}')
    model_s1 = YOLO(init_weights_s1)
    model_s1.add_callback('on_train_epoch_end', make_epoch_callback(s1_dir, EPOCHS_STAGE1))
    model_s1.train(
        data          = DATASET_DIR,
        epochs        = EPOCHS_STAGE1,
        imgsz         = IMG_SIZE,
        batch         = BATCH,
        workers       = WORKERS,
        project       = PROJECT,
        name          = RUN_STAGE1,
        optimizer     = 'AdamW',
        patience      = 25,
        lr0           = 1e-4,
        lrf           = 1e-2,
        cos_lr        = True,
        warmup_epochs = 3,
        dropout       = 0.3,
        augment       = True,
        device        = DEVICE,
        exist_ok      = True,
        resume        = resuming_s1,
        verbose       = True,
    )
    if not session_timed_out and os.path.exists(s1_best):
        save_stage_state(s1_dir, EPOCHS_STAGE1, True)
        print(f'Aşama 1 tamamlandı: {s1_best}')

BEST_STAGE1_PATH = s1_best if os.path.exists(s1_best) else s1_last

# Aşama 2: 576x576 FixRes Yüksek Çözünürlük İnce Ayarı (Fine-Tuning)
s2_dir  = os.path.join(PROJECT, RUN_STAGE2)
s2_last = os.path.join(s2_dir, 'weights', 'last.pt')
s2_best = os.path.join(s2_dir, 'weights', 'best.pt')
BEST_STAGE2_PATH = BEST_STAGE1_PATH

if is_stage_completed(s2_dir, EPOCHS_STAGE2):
    BEST_STAGE2_PATH = s2_best
    print(f'AŞAMA 2 (FixRes 576px) önceki oturumda tamamlanmış! Eğitim atlanıyor -> {s2_best}')
elif not session_timed_out and (time.time() - SESSION_START_TIME) / 3600.0 < MAX_SESSION_HOURS - 0.3:
    resuming_s2 = os.path.exists(s2_last)
    init_weights_s2 = s2_last if resuming_s2 else BEST_STAGE1_PATH
    print(f'\n Aşama 2 (FixRes 576x576) başlatılıyor (Resume={resuming_s2}): {init_weights_s2}')
    model_s2 = YOLO(init_weights_s2)
    model_s2.add_callback('on_train_epoch_end', make_epoch_callback(s2_dir, EPOCHS_STAGE2))
    model_s2.train(
        data            = DATASET_DIR,
        epochs          = EPOCHS_STAGE2,
        imgsz           = IMG_SIZE_HIRES,
        batch           = BATCH,
        workers         = WORKERS,
        project         = PROJECT,
        name            = RUN_STAGE2,
        optimizer       = 'AdamW',
        patience        = 8,
        lr0             = 2.5e-5,
        lrf             = 0.05,
        cos_lr          = True,
        warmup_epochs   = 1,
        dropout         = 0.25,
        label_smoothing = 0.03,
        augment         = True,
        device          = DEVICE,
        exist_ok        = True,
        resume          = resuming_s2,
        verbose         = True,
    )
    if not session_timed_out and os.path.exists(s2_best):
        save_stage_state(s2_dir, EPOCHS_STAGE2, True)
        BEST_STAGE2_PATH = s2_best
        print(f'Aşama 2 tamamlandı: {BEST_STAGE2_PATH}')
    elif os.path.exists(s2_best):
        BEST_STAGE2_PATH = s2_best


In [ ]:
# ==============================================================================
# Bölüm 4: Test-Time Augmentation (TTA), Çoklu Projeksiyon Füzyonu ve Pareto Kalibrasyonu
# ==============================================================================
def collect_split_files(split_name):
    files, labels, cases, views = [], [], [], []
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        cls_dir = os.path.join(DATASET_DIR, split_name, cls_name)
        if not os.path.exists(cls_dir): continue
        for fpath in sorted(glob.glob(os.path.join(cls_dir, '*.jpg'))):
            bname = os.path.basename(fpath)
            m_case = re.search(r'case(\d+)', bname)
            m_view = re.search(r'_(RCC|LCC|RMLO|LMLO)_', bname)
            files.append(fpath)
            labels.append(cls_idx)
            cases.append(int(m_case.group(1)) if m_case else -1)
            views.append(m_view.group(1) if m_view else 'UNK')
    return files, np.array(labels), np.array(cases), np.array(views)

val_files, y_val, val_cases, val_views     = collect_split_files('val')
test_files, y_true, test_cases, test_views = collect_split_files('test')

if os.path.exists(CACHE_NPZ) and is_stage_completed(s2_dir, EPOCHS_STAGE2):
    print(f'Kayıtlı TTA tahmin önbelleği yüklendi: {CACHE_NPZ}')
    cached = np.load(CACHE_NPZ)
    val_raw, val_s1_tta, val_s2_tta   = cached['val_raw'], cached['val_s1_tta'], cached['val_s2_tta']
    test_raw, test_s1_tta, test_s2_tta = cached['test_raw'], cached['test_s1_tta'], cached['test_s2_tta']
else:
    net_s1 = YOLO(BEST_STAGE1_PATH)
    net_s2 = YOLO(BEST_STAGE2_PATH) if BEST_STAGE2_PATH != BEST_STAGE1_PATH else None

    def predict_6view_tta(file_list, batch_size=64):
        n = len(file_list)
        p_s1_512      = np.zeros((n, 3), dtype=np.float64)
        p_s1_512_flip = np.zeros((n, 3), dtype=np.float64)
        p_s1_576      = np.zeros((n, 3), dtype=np.float64)
        p_s1_576_flip = np.zeros((n, 3), dtype=np.float64)
        p_s2_576      = np.zeros((n, 3), dtype=np.float64)
        p_s2_576_flip = np.zeros((n, 3), dtype=np.float64)

        for start in range(0, n, batch_size):
            batch_files = file_list[start:start+batch_size]
            imgs_bgr  = [cv2.imread(fp) for fp in batch_files]
            imgs_flip = [cv2.flip(im, 1) for im in imgs_bgr]

            for i, r in enumerate(net_s1.predict(imgs_bgr, imgsz=512, verbose=False)):
                p_s1_512[start+i] = r.probs.data.cpu().numpy()
            for i, r in enumerate(net_s1.predict(imgs_flip, imgsz=512, verbose=False)):
                p_s1_512_flip[start+i] = r.probs.data.cpu().numpy()
            for i, r in enumerate(net_s1.predict(imgs_bgr, imgsz=576, verbose=False)):
                p_s1_576[start+i] = r.probs.data.cpu().numpy()
            for i, r in enumerate(net_s1.predict(imgs_flip, imgsz=576, verbose=False)):
                p_s1_576_flip[start+i] = r.probs.data.cpu().numpy()

            if net_s2 is not None:
                for i, r in enumerate(net_s2.predict(imgs_bgr, imgsz=576, verbose=False)):
                    p_s2_576[start+i] = r.probs.data.cpu().numpy()
                for i, r in enumerate(net_s2.predict(imgs_flip, imgsz=576, verbose=False)):
                    p_s2_576_flip[start+i] = r.probs.data.cpu().numpy()
            else:
                p_s2_576[start:start+len(batch_files)]      = p_s1_576[start:start+len(batch_files)]
                p_s2_576_flip[start:start+len(batch_files)] = p_s1_576_flip[start:start+len(batch_files)]

        p_stage1_tta = 0.35 * p_s1_512 + 0.25 * p_s1_512_flip + 0.25 * p_s1_576 + 0.15 * p_s1_576_flip
        p_stage2_tta = 0.55 * p_s2_576 + 0.45 * p_s2_576_flip
        return p_s1_512, p_stage1_tta, p_stage2_tta

    print('Validation ve Test kümeleri üzerinde 6-Yönlü TTA hesaplanıyor ve önbelleğe kaydediliyor...')
    val_raw, val_s1_tta, val_s2_tta    = predict_6view_tta(val_files)
    test_raw, test_s1_tta, test_s2_tta = predict_6view_tta(test_files)
    np.savez_compressed(
        CACHE_NPZ,
        val_raw=val_raw, val_s1_tta=val_s1_tta, val_s2_tta=val_s2_tta,
        test_raw=test_raw, test_s1_tta=test_s1_tta, test_s2_tta=test_s2_tta
    )
    print(f'TTA önbelleği diske kaydedildi: {CACHE_NPZ}')

# Doğrulama kümesinde Aşama 1 ve Aşama 2 olasılık çıktıları için optimal füzyon ağırlığının (alpha) belirlenmesi
y_val_bin = label_binarize(y_val, classes=[0, 1, 2])
best_alpha, best_v_auc = 1.0, -1.0
for alpha in np.linspace(0.15, 1.0, 18):
    blend_v = alpha * val_s1_tta + (1.0 - alpha) * val_s2_tta
    v_auc = roc_auc_score(y_val_bin, blend_v, multi_class='ovr', average='macro')
    if v_auc > best_v_auc:
        best_v_auc = v_auc
        best_alpha = alpha

val_probs_tta  = best_alpha * val_s1_tta  + (1.0 - best_alpha) * val_s2_tta
test_probs_tta = best_alpha * test_s1_tta + (1.0 - best_alpha) * test_s2_tta

# Hasta Düzeyinde Çoklu Projeksiyon Bağlamsal Füzyonu (Patient-Level Multi-View Contextual Fusion)
# Klinik mamografi okuma protokollerine paralel olarak tekil projeksiyon olasılık dağılımı,
# ipsilateral, kontralateral ve bilateral projeksiyon havuzlama çıktıları ile harmanlanır.
def apply_patient_multiview_context(probs, cases, views, gamma_mv=0.45):
    """Hasta düzeyinde çoklu projeksiyon (RCC, LCC, RMLO, LMLO) bağlamsal füzyon fonksiyonu.

    Klinik mamografi değerlendirme protokollerine uygun olarak tekil görüntü olasılıklarını,
    ipsilateral, kontralateral ve bilateral projeksiyon havuzlama çıktıları ile harmanlar.
    """
    if (cases >= 0).sum() < len(cases) * 0.5:
        return probs
    out_p = probs.copy()
    df_c = pd.DataFrame({'case': cases, 'view': views, 'p0': probs[:, 0], 'p1': probs[:, 1], 'p2': probs[:, 2]})
    case_mean = df_c.groupby('case')[['p0', 'p1', 'p2']].transform('mean').values
    case_max  = df_c.groupby('case')[['p0', 'p1', 'p2']].transform('max').values

    # Yüksek riskli (BI-RADS 4-5) sınıfta yanlış negatifliği asgariye indirmek amacıyla maksimum havuzlama bileşeni ağırlıklandırılır.
    patient_context = np.zeros_like(probs)
    patient_context[:, 0] = 0.70 * case_mean[:, 0] + 0.30 * case_max[:, 0]
    patient_context[:, 1] = 0.75 * case_mean[:, 1] + 0.25 * case_max[:, 1]
    patient_context[:, 2] = 0.45 * case_mean[:, 2] + 0.55 * case_max[:, 2]
    patient_context /= np.clip(patient_context.sum(axis=1, keepdims=True), 1e-8, None)

    fused = (1.0 - gamma_mv) * out_p + gamma_mv * patient_context
    return fused / np.clip(fused.sum(axis=1, keepdims=True), 1e-8, None)

# Doğrulama kümesi üzerinde makro AUC metriğini maksimize eden bağlam ağırlığının (gamma_mv) tespiti
best_gamma, best_mv_auc = 0.0, best_v_auc
for g_cand in np.linspace(0.0, 0.75, 16):
    cand_v = apply_patient_multiview_context(val_probs_tta, val_cases, val_views, gamma_mv=g_cand)
    c_auc = roc_auc_score(y_val_bin, cand_v, multi_class='ovr', average='macro')
    if c_auc > best_mv_auc:
        best_mv_auc = c_auc
        best_gamma = g_cand

print(f'Optimal Multi-View Bağlam Ağırlığı: gamma={best_gamma:.2f} (Val Macro AUC: {best_v_auc:.4f} -> {best_mv_auc:.4f})')
val_probs_mv  = apply_patient_multiview_context(val_probs_tta, val_cases, val_views, gamma_mv=best_gamma)
test_probs_mv = apply_patient_multiview_context(test_probs_tta, test_cases, test_views, gamma_mv=best_gamma)

# Pareto-Optimal Klinik Karar Kalibrasyonu (BI-RADS 4-5 Duyarlılık ve Makro F1 Optimizasyonu)
def apply_calibration(probs, params):
    w0, w1, w2 = params
    logits = np.log(np.clip(probs, 1e-8, 1.0))
    logits[:, 0] += w0; logits[:, 1] += w1; logits[:, 2] += w2
    exp_l = np.exp(logits - logits.max(axis=1, keepdims=True))
    return exp_l / exp_l.sum(axis=1, keepdims=True)

def optimize_pareto_calibration(val_p, y_v):
    """Klinik karar eşiklerinin Pareto-optimal kalibrasyonu.

    Yüksek riskli BI-RADS 4-5 sınıfında yanlış negatifliği asgariye indirmek amacıyla
    duyarlılık (sensitivity) ile genel makro F1 skorunu eşzamanlı optimize eder.
    """
    raw_preds_v = val_p.argmax(axis=1)
    raw_rec_45  = ((raw_preds_v == 2) & (y_v == 2)).sum() / max((y_v == 2).sum(), 1)
    raw_f1_0    = f1_score(y_v, raw_preds_v, average=None)[0]

    def _obj(params):
        cal_v = apply_calibration(val_p, params)
        preds_v = cal_v.argmax(axis=1)
        f1_c = f1_score(y_v, preds_v, average=None)
        macro_f1 = f1_c.mean()
        acc = accuracy_score(y_v, preds_v)
        tp_45 = ((preds_v == 2) & (y_v == 2)).sum(); fn_45 = ((preds_v != 2) & (y_v == 2)).sum()
        tn_45 = ((preds_v != 2) & (y_v != 2)).sum(); fp_45 = ((preds_v == 2) & (y_v != 2)).sum()
        rec_45  = tp_45 / max(tp_45 + fn_45, 1)
        spec_45 = tn_45 / max(tn_45 + fp_45, 1)
        bal_45  = 0.5 * (rec_45 + spec_45)

        penalty = 0.0
        if rec_45 < raw_rec_45 + 0.015:
            penalty += 6.0 * (raw_rec_45 + 0.015 - rec_45)
        if f1_c[0] < raw_f1_0:
            penalty += 5.0 * (raw_f1_0 - f1_c[0])
        reg = 0.015 * (params[0]**2 + params[1]**2 + params[2]**2)
        return -(0.40 * macro_f1 + 0.30 * acc + 0.30 * bal_45) + penalty + reg

    best_p = [0.0, 0.0, 0.0]
    best_val = _obj(best_p)
    for iw0 in [-0.08, -0.04, 0.0, 0.04, 0.08]:
        for iw2 in [0.0, 0.05, 0.10, 0.15, 0.20]:
            res = minimize(_obj, x0=[iw0, 0.0, iw2], method='Nelder-Mead', options={'maxiter': 500})
            if res.fun < best_val:
                best_val = res.fun
                best_p = res.x
    return best_p

best_params_mv = optimize_pareto_calibration(val_probs_mv, y_val)
y_scores = apply_calibration(test_probs_mv, best_params_mv)
y_pred   = y_scores.argmax(axis=1)
acc      = (y_true == y_pred).mean() * 100

print(f'Klinik Kalibrasyon Ağırlıkları: {[round(x, 4) for x in best_params_mv]}')
print('\n Sınıflandırma Raporu (Multi-View Context + 6-View TTA + Clinical Calibration):')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))
print(f'Test Doğruluğu: {acc:.2f}%')


In [ ]:
# ==============================================================================
# Bölüm 5: Ablasyon Analizi ve Performans Değerlendirmesi (Tablo 4)
# ==============================================================================
def compute_paper_table4_metrics(y_t, probs, preds=None):
    """Makale Tablo 4 için tanısal ve istatistiksel performans metriklerini hesaplar.

    Metrikler: Doğruluk, Makro/Sınıf Bazlı Hassasiyet, Duyarlılık, F1-Skoru, Makro ROC-AUC
    ve klinik sevk odaklı BI-RADS 4-5 duyarlılık oranı.
    """
    if preds is None:
        preds = probs.argmax(axis=1)
    y_bin = label_binarize(y_t, classes=[0, 1, 2])
    auc_0  = roc_auc_score(y_bin[:, 0], probs[:, 0])
    auc_12 = roc_auc_score(y_bin[:, 1], probs[:, 1])
    auc_45 = roc_auc_score(y_bin[:, 2], probs[:, 2])
    auc_macro = roc_auc_score(y_bin, probs, multi_class='ovr', average='macro')

    diag_mask = (y_t != 0) & (preds != 0)
    diag_acc = (y_t[diag_mask] == preds[diag_mask]).mean() * 100.0 if diag_mask.sum() > 0 else 0.0

    tp_45 = ((preds == 2) & (y_t == 2)).sum(); fn_45 = ((preds != 2) & (y_t == 2)).sum()
    tn_45 = ((preds != 2) & (y_t != 2)).sum(); fp_45 = ((preds == 2) & (y_t != 2)).sum()
    rec_45  = (tp_45 / max(tp_45 + fn_45, 1)) * 100.0
    spec_45 = (tn_45 / max(tn_45 + fp_45, 1)) * 100.0
    prec_45 = (tp_45 / max(tp_45 + fp_45, 1)) * 100.0
    f1_c = f1_score(y_t, preds, average=None)

    return {
        'BI-RADS 4-5 ROC-AUC': round(auc_45, 4),
        'BI-RADS 1-2 ROC-AUC': round(auc_12, 4),
        'BI-RADS 0 ROC-AUC': round(auc_0, 4),
        'ROC-AUC (Macro)': round(auc_macro, 4),
        '2x2 Tanısal Alt-Matris Doğruluğu (1-2 vs 4-5) (%)': round(diag_acc, 2),
        'BI-RADS 4-5 Dengeli Doğruluk (%)': round(0.5 * (rec_45 + spec_45), 2),
        'BI-RADS 4-5 Özgüllük / Specificity (%)': round(spec_45, 2),
        'BI-RADS 4-5 Kesinlik / Precision (%)': round(prec_45, 2),
        'BI-RADS 4-5 Duyarlılık / Recall (%)': round(rec_45, 2),
        'BI-RADS 4-5 F1-Skoru': round(f1_c[2], 4),
        'BI-RADS 1-2 F1-Skoru': round(f1_c[1], 4),
        'BI-RADS 0 F1-Skoru': round(f1_c[0], 4),
        '3-Sınıflı Genel Doğruluk (Test Accuracy) (%)': round((y_t == preds).mean() * 100.0, 2),
        'Makro F1-Skoru (3-Sınıf)': round(f1_score(y_t, preds, average='macro'), 4),
    }

paper_baseline = {
    "BI-RADS 4-5 ROC-AUC": 0.809,
    "BI-RADS 1-2 ROC-AUC": 0.779,
    "BI-RADS 0 ROC-AUC": 0.718,
    "ROC-AUC (Macro)": 0.769,
    "2x2 Tan\u0131sal Alt-Matris Do\u011frulu\u011fu (1-2 vs 4-5) (%)": 77.19,
    "BI-RADS 4-5 Dengeli Do\u011fruluk (%)": 71.8,
    "BI-RADS 4-5 \u00d6zg\u00fcll\u00fck / Specificity (%)": 86.22,
    "BI-RADS 4-5 Kesinlik / Precision (%)": 72.19,
    "BI-RADS 4-5 Duyarl\u0131l\u0131k / Recall (%)": 57.37,
    "BI-RADS 4-5 F1-Skoru": 0.64,
    "BI-RADS 1-2 F1-Skoru": 0.65,
    "BI-RADS 0 F1-Skoru": 0.42,
    "3-S\u0131n\u0131fl\u0131 Genel Do\u011fruluk (Test Accuracy) (%)": 59.4,
    "Makro F1-Skoru (3-S\u0131n\u0131f)": 0.57
}

comp_df = pd.DataFrame({
    'Makale Baz (YOLO11m-cls)': paper_baseline,
    'Yeni: Stage-1 (Tekil)': compute_paper_table4_metrics(y_true, test_raw),
    'Yeni: 6-Yön TTA': compute_paper_table4_metrics(y_true, test_probs_tta),
    'Yeni: Multi-View + Kalibre': compute_paper_table4_metrics(y_true, y_scores, y_pred),
})
comp_df['Kazanç (Yeni - Baz)'] = (comp_df['Yeni: Multi-View + Kalibre'] - comp_df['Makale Baz (YOLO11m-cls)']).round(4)

print('\n' + '=' * 110)
print('TABLO 4 — KARŞILAŞTIRMALI PERFORMANS ANALİZİ (YOLO11m-cls)')
print('=' * 110)
print(comp_df.to_string())
print('=' * 110)
comp_df.to_csv(os.path.join(PROJECT, 'yolo11_table4_comparison.csv'))


In [ ]:
# ==============================================================================
# Bölüm 6: Tanısal Performans Analizi, Hata Matrisi ve ROC Eğrileri
# ==============================================================================
COLORS = ['#6C63FF', '#FF6584', '#43C59E']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#1a1a2e')

ax = axes[0]
ax.set_facecolor('#16213e')
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(DISPLAY_NAMES, fontsize=11, color='white', rotation=15)
ax.set_yticklabels(DISPLAY_NAMES, fontsize=11, color='white')
ax.set_xlabel('Tahmin Edilen', fontsize=12, color='white')
ax.set_ylabel('Gerçek', fontsize=12, color='white')
ax.set_title('Confusion Matrix (Normalize)', fontsize=14, color='white', pad=15)
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_edgecolor('#444')
for i in range(3):
    for j in range(3):
        color = 'white' if cm_norm[i, j] < 0.5 else '#1a1a2e'
        ax.text(j, i, f'{cm_norm[i,j]:.2f}\n({cm[i,j]})',
                ha='center', va='center', fontsize=10, color=color, fontweight='bold')

ax2 = axes[1]
ax2.set_facecolor('#16213e')
y_bin = label_binarize(y_true, classes=[0, 1, 2])
for i, (cls_name, color) in enumerate(zip(DISPLAY_NAMES, COLORS)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_scores[:, i])
    roc_auc = auc(fpr, tpr)
    ax2.plot(fpr, tpr, color=color, lw=2.5, label=f'{cls_name} (AUC = {roc_auc:.3f})')
ax2.plot([0, 1], [0, 1], 'w--', lw=1, alpha=0.5)
ax2.set_xlim([0.0, 1.0]); ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('False Positive Rate', fontsize=12, color='white')
ax2.set_ylabel('True Positive Rate', fontsize=12, color='white')
ax2.set_title('ROC Eğrileri (One-vs-Rest)', fontsize=14, color='white', pad=15)
ax2.tick_params(colors='white')
for spine in ax2.spines.values(): spine.set_edgecolor('#444')
ax2.legend(loc='lower right', fontsize=11, facecolor='#1a1a2e', labelcolor='white', edgecolor='#555')

plt.suptitle(f'YOLO11m-cls — MammosighTR BI-RADS Sınıflandırma | Test Doğruluğu: {acc:.2f}%',
             fontsize=15, color='white', y=1.02, fontweight='bold')
plt.tight_layout()
out_fig = '/kaggle/working/yolo11_results.png' if os.path.exists('/kaggle') else './yolo11_results.png'
plt.savefig(out_fig, dpi=180, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

results_csv = os.path.join(s1_dir, 'results.csv')
if os.path.exists(results_csv):
    df_res = pd.read_csv(results_csv)
    df_res.columns = df_res.columns.str.strip()
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#1a1a2e')
    ax = axes[0]; ax.set_facecolor('#16213e')
    for col in [c for c in df_res.columns if 'loss' in c.lower()]:
        ax.plot(df_res['epoch'], df_res[col], label=col, lw=2)
    ax.set_xlabel('Epoch', color='white'); ax.set_ylabel('Loss', color='white')
    ax.set_title('Eğitim / Doğrulama Loss', color='white', fontsize=13)
    ax.tick_params(colors='white'); ax.legend(facecolor='#1a1a2e', labelcolor='white', edgecolor='#555')
    for spine in ax.spines.values(): spine.set_edgecolor('#444')

    ax2 = axes[1]; ax2.set_facecolor('#16213e')
    for col in [c for c in df_res.columns if 'acc' in c.lower() or 'top' in c.lower()]:
        ax2.plot(df_res['epoch'], df_res[col], label=col, lw=2)
    ax2.set_xlabel('Epoch', color='white'); ax2.set_ylabel('Accuracy', color='white')
    ax2.set_title('Top-1 / Top-5 Accuracy', color='white', fontsize=13)
    ax2.tick_params(colors='white'); ax2.legend(facecolor='#1a1a2e', labelcolor='white', edgecolor='#555')
    for spine in ax2.spines.values(): spine.set_edgecolor('#444')

    plt.suptitle('YOLO11m-cls — Eğitim Eğrileri', color='white', fontsize=14, fontweight='bold')
    plt.tight_layout()
    curve_fig = '/kaggle/working/yolo11_training_curves.png' if os.path.exists('/kaggle') else './yolo11_training_curves.png'
    plt.savefig(curve_fig, dpi=180, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
